steps
1. get the data
2. separate the target and features
3. split the data : testing and training
4. preprocess the data
5. fit the model
6. predict
7. evaluate


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from xgboost import callback
from sklearn.feature_selection import mutual_info_regression



In [2]:
#get the data
house_data =pd.read_csv("/kaggle/input/competitions/home-data-for-ml-course/train.csv")
house_data.head()


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
#separate target and features
x =house_data.drop("SalePrice",axis=1)
y= house_data.SalePrice

In [4]:
x_copy =x.copy()
y_copy =y.copy()
num_cols =x_copy.select_dtypes(include =["number"]).columns
cat_cols=x_copy.select_dtypes(include=["object"]).columns


num_train_columns =x_copy[num_cols]
cat_train_columns =x_copy[cat_cols]

num_transformer=SimpleImputer(strategy="mean")
cat_transformer =SimpleImputer(strategy="most_frequent")

imputed_num=pd.DataFrame(num_transformer.fit_transform(x_copy[num_cols]))
imputed_cat =pd.DataFrame(cat_transformer.fit_transform(x_copy[cat_cols]))



In [5]:
ordinal_Encoder=OrdinalEncoder()
encoded_cat =pd.DataFrame(ordinal_Encoder.fit_transform(imputed_cat))

In [6]:
imputed_num.columns =num_train_columns.columns
encoded_cat.columns = cat_train_columns.columns

In [7]:
x_copy=pd.concat([imputed_num,encoded_cat],axis=1)


In [8]:
#getting mutual information
discrete_features =x_copy.dtypes == int

def make_mi_scores(x_copy,y_copy,discrete_features):
    mi_scores =mutual_info_regression(x_copy,y_copy,discrete_features =discrete_features)
    mi_scores=pd.Series(mi_scores,name="Mi_scores",index=x.columns)
    mi_scores=mi_scores.sort_values(ascending=False)
    return mi_scores
    
    

In [9]:
mi_scores =make_mi_scores(x_copy,y_copy,discrete_features)

In [10]:
mi_scores.head(10)


LotArea         0.565064
LowQualFinSF    0.502701
HouseStyle      0.483068
Neighborhood    0.366281
MasVnrArea      0.364083
Alley           0.362691
ExterQual       0.360500
OpenPorchSF     0.339938
Functional      0.319633
GarageType      0.309605
Name: Mi_scores, dtype: float64

In [11]:
features =["OverallQual","Neighborhood","GrLivArea","GarageCars","TotalBsmtSF","GarageArea","YearBuilt","KitchenQual","BsmtQual","ExterQual"]

In [12]:
x=x[features]



In [13]:
#split the data
train_x,val_x,train_y,val_y=train_test_split(x,y,train_size=0.8,test_size=0.2,random_state=1)

In [14]:
#preprocess the data
numeric_col =train_x.select_dtypes(include=["number"]).columns
categorical_col =train_x.select_dtypes(exclude=["number"]).columns
print(categorical_col)


#preprocess numeric data
numerical_transformer =SimpleImputer()

#preprocess categorical data
categorical_transformer =Pipeline(steps =[("imputer",SimpleImputer(strategy ="most_frequent")),
                                           ("ordinal",
                                            OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))])

#bundle together numerical and categorical preprocessing
preprocessing =ColumnTransformer(transformers=[("num",numerical_transformer,numeric_col ),("cat",categorical_transformer,categorical_col)])

Index(['Neighborhood', 'KitchenQual', 'BsmtQual', 'ExterQual'], dtype='object')


In [15]:
#create model
house_model =XGBRegressor(n_estimators=500,early_stopping_rounds=5,learning_rate=0.05,random_state=1)

In [16]:
#bundle together preprocessing and modellling code
my_pipeline =Pipeline(steps =[("preprocessing",preprocessing),("model",house_model)])


In [17]:
my_pipeline.named_steps["preprocessing"].fit(train_x)
val_x_processed= my_pipeline.named_steps["preprocessing"].transform(val_x)

#train the model
my_pipeline.fit(train_x,train_y,model__eval_set=[(val_x_processed,val_y)],model__verbose=False)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', SimpleImputer(),
                                                  Index(['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'GarageArea',
       'YearBuilt'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinal',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  In...
                              feature_weights=None, gamma=None,
                              grow_policy=None, importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [18]:
preds =my_pipeline.predict(val_x)

In [19]:
mean_absolute_error(val_y,preds)

17625.986328125

In [20]:
best_iteration =my_pipeline.named_steps["model"].best_iteration

In [21]:
final_model=XGBRegressor(n_estimators=best_iteration,random_state=1)

In [22]:
final_pipeline=Pipeline(steps=[("preprocessing",preprocessing),("model",final_model)])

In [23]:
final_pipeline.fit(x,y)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', SimpleImputer(),
                                                  Index(['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'GarageArea',
       'YearBuilt'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinal',
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  In...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=146, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [24]:
test_data =pd.read_csv("/kaggle/input/competitions/home-data-for-ml-course/test.csv")

In [25]:
test_preds =final_pipeline.predict(test_data)

In [26]:
output=pd.DataFrame({"Id":test_data.Id,"SalePrice":test_preds})

In [27]:
output.to_csv("submission.csv",index=False)